# Week A — see the data

**Corrected version.** Three bugs from the first run are fixed here:

1. PRIDE returns `ftp://` links; `requests` only speaks HTTP — now converted.
2. `low_memory` and `engine="python"` cannot be combined — now uses an explicit tab separator.
3. The column filter omitted `gene`, hiding `Gene names` entirely — which silently caused every gene lookup in Week B to fail.

**Goal:** open a real PRIDE deposit and answer one question — *can I tell which column is which experimental condition?*

Run cells with `Shift + Enter`, top to bottom.

## Step 0 — browser first, no code

Open `https://www.ebi.ac.uk/pride/archive/projects/PXD018299` and note:

1. How many files?
2. Processed search results, or only `.raw`?
3. Any SDRF (`.sdrf.tsv`)?
4. Do filenames encode conditions?

**Note:** the 2025 TNBC accessions (PXD064305 and others) are still private pending publication. PRIDE reserves the identifier at submission and releases on publication.

In [ ]:
import pandas as pd
import requests
import os

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("pandas", pd.__version__)

## Step 1 — list the deposit's files

Tries the current PRIDE API and falls back to the older one. If both fail, skip to Step 3b.

In [ ]:
ACCESSION = "PXD018299"

def list_pride_files(accession):
    for url in [
        f"https://www.ebi.ac.uk/pride/ws/archive/v3/projects/{accession}/files",
        f"https://www.ebi.ac.uk/pride/ws/archive/v2/files/byProject?accession={accession}",
    ]:
        try:
            r = requests.get(url, timeout=90)
            if r.status_code == 200:
                data = r.json()
                items = data if isinstance(data, list) else data.get("_embedded", {}).get("files", [])
                if items:
                    print("worked:", url)
                    return items
        except Exception as e:
            print("failed:", url, "->", e)
    return []

raw_items = list_pride_files(ACCESSION)
print(f"{len(raw_items)} files found")

## Step 2 — readable table of files

**Fix 1.** PRIDE returns `ftp://` addresses, which `requests` cannot open — it raises `InvalidSchema: No connection adapters were found`. The conversion to `https://ftp.pride.ebi.ac.uk` happens as the table is built.

This is permanent adapter behaviour, not a Colab workaround.

In [ ]:
def to_https(url):
    if url and url.startswith("ftp://"):
        return url.replace("ftp://ftp.pride.ebi.ac.uk", "https://ftp.pride.ebi.ac.uk")
    return url

def tidy(items):
    rows = []
    for f in items:
        link = None
        for loc in f.get("publicFileLocations", []):
            v = loc.get("value", "")
            if v.startswith("http"):
                link = v
                break
            link = link or v
        rows.append({
            "name": f.get("fileName"),
            "mb": round((f.get("fileSizeBytes") or 0) / 1e6, 1),
            "link": to_https(link),
        })
    return pd.DataFrame(rows).sort_values("mb").reset_index(drop=True)

files = tidy(raw_items)
print(f"total files: {len(files)}\n")
print("extensions present:")
print(files["name"].str.rsplit(".", n=1).str[-1].value_counts())
print("\nsmallest 25:")
files.head(25)[["name", "mb"]]

## Step 3 — download one processed file

Never a `.raw` file — those are tens of gigabytes of instrument output and unusable here. For PXD018299 the target is `HAP1_USP18KO_GlyGlyKSites.txt`, about 2.8 MB, and it contains everything the paper's figures came from.

In [ ]:
PICK = "GlyGlyKSites"

candidates = files[files["name"].str.contains(PICK, case=False, na=False)]
print(candidates[["name", "mb"]].to_string())

chosen = candidates.iloc[0]
LOCAL = chosen["name"]
print(f"\nchosen: {LOCAL} ({chosen['mb']} MB)")

if not os.path.exists(LOCAL):
    with requests.get(chosen["link"], stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(LOCAL, "wb") as fh:
            for chunk in r.iter_content(chunk_size=1 << 20):
                fh.write(chunk)

print(f"downloaded — {round(os.path.getsize(LOCAL)/1e6, 1)} MB")

### Step 3b — fallback if the API failed

Copy a link from the PRIDE website, paste below, uncomment by deleting each `#`.

In [ ]:
# URL = to_https("paste the link here")
# LOCAL = URL.rsplit("/", 1)[-1]
# with requests.get(URL, stream=True, timeout=600) as r:
#     r.raise_for_status()
#     with open(LOCAL, "wb") as fh:
#         for chunk in r.iter_content(chunk_size=1 << 20):
#             fh.write(chunk)
# print("downloaded", LOCAL)

## Step 4 — peek at raw text before parsing

Always look at the first lines as plain text. It shows the separator and whether junk sits above the real header — the most common cause of a confusing parse.

In [ ]:
with open(LOCAL, "r", errors="replace") as fh:
    for i, line in enumerate(fh):
        print(repr(line[:300]))
        if i >= 3:
            break

## Step 5 — load it

**Fix 2.** The original used `sep=None, engine="python", low_memory=False`, which raises `ValueError: The 'low_memory' option is not supported with the 'python' engine` — `low_memory` exists only on the fast C parser, and `engine="python"` switches away from it.

MaxQuant tables are always tab-separated, so state it rather than asking pandas to guess. Your real adapter should do the same.

In [ ]:
df = pd.read_csv(LOCAL, sep="\t", low_memory=False)
print(f"rows: {len(df):,}")
print(f"columns: {len(df.columns)}")

## Step 6 — find the columns that matter

**Fix 3.** The original pattern list omitted `gene`, so `Gene names` never appeared in the output — and that silently caused every gene lookup in Week B to return "not found" fourteen times, which reads like a real negative result.

Of 159 columns, about eight matter.

In [ ]:
keep = ["protein", "gene", "position", "localization",
        "sequence", "intensity", "ratio", "reverse", "contaminant"]

for c in df.columns:
    if any(k in c.lower() for k in keep):
        print(" ", c)

In [ ]:
print("four namespaces for the same entity:\n")
for c in ["Proteins", "Protein", "Protein names", "Gene names"]:
    if c in df.columns:
        print(f"--- {c} ---")
        print(df[c].dropna().head(3).to_string(), "\n")

`Proteins` holds semicolon-separated UniProt accessions. `Protein` is the search engine's single razor pick — often a TrEMBL entry rather than the reviewed Swiss-Prot one. `Gene names` holds the symbols you actually think in.

Translating between these is not a convenience. Without it you cannot ask about "ADAR" and reach a row keyed on `P55265`.

## Step 7 — filter and measure

Decoys and contaminants go first. Then count how often one site maps to several proteins — the number that changed the schema.

In [ ]:
clean = df[(df["Reverse"] != "+") & (df["Potential contaminant"] != "+")].copy()

print(f"before filtering: {len(df):,}")
print(f"after filtering:  {len(clean):,}")

print("\nlocalization prob:")
print(clean["Localization prob"].describe())

shared = clean["Proteins"].astype(str).str.contains(";").sum()
print(f"\nsites mapping to >1 protein: {shared:,} of {len(clean):,} "
      f"({100*shared/len(clean):.0f}%)")

## Step 8 — the curation record

**The point of the week.** No SDRF accompanies this deposit, so the experimental design has to be reconstructed from the publication's methods and the column names.

That reconstruction is an inference, and invariant I8 requires it recorded with its basis rather than silently applied.

In [ ]:
import json

def sample(genotype, treatment, rep, tp):
    return {"genotype": genotype, "treatment": treatment, "timepoint_h": tp,
            "replicate": rep, "replicate_type": "unspecified",
            "cell_line": "HAP1", "organism_taxid": 9606}

IFN = "IFN-alpha2b_1000U_per_mL"
mapping = {}
for i in (1, 2, 3):
    mapping[f"Ratio mod/base WT_{i}"] = sample("WT", "none", i, None)
    mapping[f"Ratio mod/base WT_IFN_{i}"] = sample("WT", IFN, i, 48)
    mapping[f"Ratio mod/base KO_{i}"] = sample("USP18-/-", "none", i, None)
    mapping[f"Ratio mod/base KO_IFN_{i}"] = sample("USP18-/-", IFN, i, 48)

mapping["Ratio mod/base KO_1_181212063719"] = mapping.pop("Ratio mod/base KO_1")
mapping["Ratio mod/base KO_1_181212063719"]["note"] = \
    "trailing 181212063719 is an instrument run ID, not a condition"

curation = {
    "accession": ACCESSION,
    "file": LOCAL,
    "modality": "digly_proteomics",
    "search_engine": "maxquant",
    "search_engine_version": "1.5.5.1",
    "acquisition_mode": "dda",
    "instrument": "Q Exactive HF",
    "basis": "publication_methods",
    "confidence": "inferred",
    "curated_by": None,
    "rationale": (
        "Design from methods of Pinto-Fernandez et al., Br J Cancer 124:817-830 "
        "(2021), doi:10.1038/s41416-020-01167-y. HAP1 WT and USP18-/- CRISPR KO, "
        "IFN-alpha2b 1000 U/mL, GlyGly comparison at 48 h. Column names encode "
        "genotype, treatment and replicate unambiguously. No SDRF accompanies the "
        "deposit, so this is inferred from the publication rather than stated by "
        "the submitters."
    ),
    "unresolved": [
        "Timepoint for unstimulated arms not stated; left null rather than assumed.",
        "Replicate type (biological vs technical) not stated for the GlyGly peptidome.",
    ],
    "mapping": mapping,
}

missing = [k for k in mapping if k not in clean.columns]
print("columns in mapping but not in file:", missing or "none")

out = f"curation_{ACCESSION}.json"
with open(out, "w") as fh:
    json.dump(curation, fh, indent=2)
print(f"\nwrote {out} — {len(mapping)} samples mapped")
print("DOWNLOAD IT from the folder icon on the left. Colab deletes everything on exit,")
print("and under invariant I9 this is the only file that cannot be regenerated.")

## Step 9 — record what you learned

Into the **Measured findings** table in `ROADMAP.md`:

1. Which search engine, and how you could tell from the columns alone — that is your adapter's `sniff()`.
2. Whether an SDRF was present.
3. How long the design reconstruction actually took, against the one-day budget.
4. Anything the file contained that the ontology has no place for.

Point 4 is the valuable one. Both `Ratio mod/base` and the 82% multi-mapping came from this step.

**Next:** Week B — reproduce the published volcano.